## Fraud Detection

In [ ]:
# import shutil
# import os
# import kagglehub
# from pathlib import Path

# downloaded_path = kagglehub.dataset_download(
#     "neharoychoudhury/credit-card-fraud-data"
# )

# print("Downloaded to:", downloaded_path)


# destination = Path("./data./credit_card_fraud")

# destination.mkdir(parents=True, exist_ok=True)

# shutil.copytree(downloaded_path, destination, dirs_exist_ok=True)

# print("Copied to:", destination.resolve())


100%|██████████| 839k/839k [00:01<00:00, 774kB/s]

Extracting files...
Downloaded to: C:\Users\nguye\.cache\kagglehub\datasets\neharoychoudhury\credit-card-fraud-data\versions\1
Copied to: E:\scikit\scikit_learn\Machine Learning Reset\project-5-fraud-detection\data\credit_card_fraud


## Importing Important Modules

In [4]:
# number and data modules
import pandas as pd
import numpy as np

# import visualization modules
import matplotlib.pyplot as plt
import seaborn as sns

# import sklearn modules
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# import metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve
)

In [6]:
# Load dataset

df = pd.read_csv("./data/credit_card_fraud/fraud_data.csv")
df.head()

,trans_date_trans_time,merchant,category,amt,city,state,lat,long,city_pop,job,dob,trans_num,merch_lat,merch_long,is_fraud
0,04-01-2019 00:58,"""Stokes, Christiansen and Sipes""",grocery_net,14.37,Wales,AK,64.7556,-165.6723,145,"""Administrator, education""",09-11-1939,a3806e984cec6ac0096d8184c64ad3a1,65.654142,-164.722603,1
1,04-01-2019 15:06,Predovic Inc,shopping_net,966.11,Wales,AK,64.7556,-165.6723,145,"""Administrator, education""",09-11-1939,a59185fe1b9ccf21323f581d7477573f,65.468863,-165.473127,1
2,04-01-2019 22:37,Wisozk and Sons,misc_pos,49.61,Wales,AK,64.7556,-165.6723,145,"""Administrator, education""",09-11-1939,86ba3a888b42cd3925881fa34177b4e0,65.347667,-165.914542,1
3,04-01-2019 23:06,Murray-Smitham,grocery_pos,295.26,Wales,AK,64.7556,-165.6723,145,"""Administrator, education""",09-11-1939,3a068fe1d856f0ecedbed33e4b5f4496,64.445035,-166.080207,1
4,04-01-2019 23:59,Friesen Lt,health_fitness,18.17,Wales,AK,64.7556,-165.6723,145,"""Administrator, education""",09-11-1939,891cdd1191028759dc20dc224347a0ff,65.447094,-165.446843,1


In [7]:
df.is_fraud.value_counts()

is_fraud
0                         12600
1                          1844
1"2020-12-24 16:56:24"        1
0"2019-01-01 00:00:44"        1
Name: count, dtype: int64

In [8]:
# we can see that there are a huge different between the target columns of the dataset, within this dataset, the ammount of non fraud almost 9 time more than fraud
# now, lets dive into the dataset features and number

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14446 entries, 0 to 14445
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trans_date_trans_time  14446 non-null  object 
 1   merchant               14446 non-null  object 
 2   category               14446 non-null  object 
 3   amt                    14446 non-null  float64
 4   city                   14446 non-null  object 
 5   state                  14446 non-null  object 
 6   lat                    14446 non-null  float64
 7   long                   14446 non-null  float64
 8   city_pop               14446 non-null  int64  
 9   job                    14446 non-null  object 
 10  dob                    14446 non-null  object 
 11  trans_num              14446 non-null  object 
 12  merch_lat              14446 non-null  float64
 13  merch_long             14446 non-null  float64
 14  is_fraud               14446 non-null  object 
dtypes:

In [9]:
df.shape

(14446, 15)

In [10]:
df.describe()

,amt,lat,long,city_pop,merch_lat,merch_long
count,14446.000000,14446.000000,14446.000000,1.444600e+04,14446.000000,14446.000000
mean,124.430073,39.787692,-110.874225,1.065370e+05,39.787991,-110.874892
std,231.352587,5.317039,12.985813,2.902916e+05,5.360593,12.995596
min,1.000000,20.027100,-165.672300,4.600000e+01,19.032689,-166.670685
25%,12.080000,36.715400,-120.415800,4.930000e+02,36.794655,-120.146253
50%,51.520000,39.666200,-111.098500,1.645000e+03,39.620953,-111.192629
75%,101.030000,41.940400,-101.136000,3.543900e+04,42.275740,-100.446822
max,3261.470000,66.693300,-89.628700,2.383912e+06,67.510267,-88.646366


In [21]:
df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"],
    format="%d-%m-%Y %H:%M"
)

df["dob"] = pd.to_datetime(
    df["dob"],
    format="%d-%m-%Y"
)

In [22]:
df = df.drop([
    "trans_date_trans_time",
    "dob",
    "trans_num",
    "merchant",   # high cardinality
    "job"         # optional drop
], axis=1)

## Scaling

In [23]:
df.head()

,category,amt,city,state,lat,long,city_pop,merch_lat,merch_long,is_fraud
0,grocery_net,14.37,Wales,AK,64.7556,-165.6723,145,65.654142,-164.722603,1
1,shopping_net,966.11,Wales,AK,64.7556,-165.6723,145,65.468863,-165.473127,1
2,misc_pos,49.61,Wales,AK,64.7556,-165.6723,145,65.347667,-165.914542,1
3,grocery_pos,295.26,Wales,AK,64.7556,-165.6723,145,64.445035,-166.080207,1
4,health_fitness,18.17,Wales,AK,64.7556,-165.6723,145,65.447094,-165.446843,1


In [24]:
X = df.drop("is_fraud", axis=1)
y = df.is_fraud

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Define columns types for Preprocessing Pipeline

In [27]:
num_cols = X.select_dtypes(include=["int64, float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

In [28]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

## Logistic Regression

In [29]:
from sklearn.pipeline import Pipeline

log_model = Pipeline([
    ("processor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

## Random Forest

In [30]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

## Evaluation

In [34]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(name, y_true, y_pred):
    print(name)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average= "weighted"))
    print("Recall:", recall_score(y_true, y_pred,average="weighted"))
    print("F1:", f1_score(y_true, y_pred, average= "weighted"))
    print("-"*30)

evaluate("Logistic Regression", y_test, y_pred_log)
evaluate("Random Forest", y_test, y_pred_rf)

Logistic Regression
Accuracy: 0.8830449826989619
Precision: 0.8762537673329197
Recall: 0.8830449826989619
F1: 0.8467966895777109
------------------------------
Random Forest
Accuracy: 0.8685121107266436
Precision: 0.84003007135473
Recall: 0.8685121107266436
F1: 0.8450439091825676
------------------------------


c:\Users\nguye\miniconda3\envs\testenv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nguye\miniconda3\envs\testenv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
